# 02 · Build the public release from the archive

Turns `ARCHIVE/02_data/` into a release that can be joined correctly. Three things
this fixes, all of which are currently landmines for anyone downloading the data:

1. **`item_id` collides across audits.** `IT0001`–`IT0442` (audit 2) and
   `IT0000`–`IT0238` (audit 3) share 238 identifiers denoting different items. Joining
   on them produces plausible-looking garbage. This build renamespaces to `A1-`/`A2-`/`A3-`
   and introduces a single corpus primary key.
2. **Annotators saw truncated text.** Audit 1 saw 500 characters, audit 2 saw 700,
   audit 3 saw the full response. So an audit row cannot be matched to a corpus row by
   equality — it has to be matched by prefix, against the right candidate set.
3. **Two scorers and two grader prompts are in play.** These are tagged explicitly
   rather than left for a reader to discover.

**Run:** set `ARCHIVE_ROOT` (default `../ARCHIVE`) and `RELEASE_DIR`. CPU only, about a
minute. Requires only pandas and numpy.

In [1]:
import os, re, json, hashlib, collections
import numpy as np, pandas as pd

ROOT=os.environ.get("ARCHIVE_ROOT","../ARCHIVE"); DATA=os.path.join(ROOT,"02_data")
OUT =os.environ.get("RELEASE_DIR","./register-safety-gap-v2")
def p(*a): return os.path.join(DATA,*a)
for d in ["prompts","generations","grader","audits","meta"]: os.makedirs(os.path.join(OUT,d),exist_ok=True)

## 1 · Naming conventions

The corpus primary key is `response_id = {model_slug}__{pair_id}__{framing}`. It is
unique across the 3,600 responses, stable across all three generation runs, and
readable, which matters when someone is eyeballing a join that went wrong.

`eval_framing`/`deploy_framing` are renamed to `benchmark`/`user`. The originals were
never accurate — nothing was deployed — and the paper's own framing contrast is
benchmark-style versus user-style prose.

In [2]:
SLUG={"Qwen/Qwen2.5-0.5B-Instruct":"qwen2.5-0.5b","Qwen/Qwen2.5-1.5B-Instruct":"qwen2.5-1.5b",
      "Qwen/Qwen2.5-3B-Instruct":"qwen2.5-3b","Qwen/Qwen2.5-7B-Instruct":"qwen2.5-7b",
      "google/gemma-2-2b-it":"gemma2-2b","google/gemma-2-9b-it":"gemma2-9b"}
FR={"eval_framing":"benchmark","deploy_framing":"user"}
LAB={"HEDGE":"HEDGE","REFUSE":"REFUSAL","REFUSAL":"REFUSAL","COMPLY":"HARMFUL","HARMFUL":"HARMFUL"}
def rid(m,pi,f): return f"{SLUG[m]}__{pi}__{FR[f]}"
def norm(t): return re.sub(r"\s+"," ",str(t)).strip()
def lab(x): return LAB[str(x).strip().upper()]

## 2 · Load the source artifacts

In [3]:
pairs=pd.read_csv(p("original","register_pairs_300.csv"))
gen  =pd.concat([pd.read_csv(p("original","qwen_generations.csv")),
                 pd.read_csv(p("original","gemma_generations.csv"))],ignore_index=True)
judge=pd.read_csv(p("original","judge_v2_rescored.csv"))
guard=pd.read_csv(p("original","second_grader_labels.csv"))
gen["nr"]=gen.reply.map(norm)
gen["response_id"]=[rid(m,pi,f) for m,pi,f in zip(gen.model,gen.pair_id,gen.framing)]

## 3 · The resolver

Each audit gives different side information, so each gets a different candidate set:

| Audit | Side information available | Candidate set |
|---|---|---|
| 1 | model, framing | responses from that model under that framing |
| 2 | framed prompt (→ pair_id), framing | responses to that prompt |
| 3 | intent, full response text | responses to any prompt with that intent |

Within the candidate set, match by prefix on the normalised text the annotator actually
saw, dropping the last few characters to absorb a word cut mid-token by the display
window.

Where more than one response shares the prefix — usually identical canned refusals
emitted by several models — the item is recorded as `ambiguous_k` rather than assigned
a guess. That is the honest outcome and it is why `link_status` is a released column.

In [4]:
# ---------------------------------------------------------------- resolver
BY_PF=collections.defaultdict(list)          # (pair_id, framing) -> [(rid, full_text)]
BY_MF=collections.defaultdict(list)          # (model, framing)   -> [(rid, full_text)]
BY_IN=collections.defaultdict(list)          # intent             -> [(rid, framing, full_text)]
intent=dict(zip(pairs.pair_id,pairs.intent))
for t in gen.itertuples():
    BY_PF[(t.pair_id,t.framing)].append((t.response_id,t.nr))
    BY_MF[(t.model,t.framing)].append((t.response_id,t.nr))
    BY_IN[intent[t.pair_id]].append((t.response_id,t.framing,t.nr))
PROMPT2PAIR={(norm(r),f):pid for f,col in [("eval_framing","eval_framing"),("deploy_framing","deploy_framing")]
             for pid,r in zip(pairs.pair_id,pairs[col])}

def _grade_set(cands):
    """Grader label is recoverable whenever every candidate response carries the same one."""
    c={gl[r] for r in cands if r in gl}
    return list(c)[0] if len(c)==1 else None


def _pick(cands):
    if len(cands)==1: return cands[0],"unique"
    if len(cands)==0: return None,"unmatched"
    return None,f"ambiguous_{len(cands)}"

def resolve(shown,*,pair_id=None,framing=None,model=None,intent_text=None):
    """Prefix-match the (possibly window-truncated) text the annotator saw."""
    s=norm(shown).rstrip(); s=s[:max(len(s)-3,1)]      # drop trailing partial word
    if pair_id is not None and framing is not None:
        return _pick([r for r,f in BY_PF[(pair_id,framing)] if f.startswith(s)])
    if model is not None and framing is not None:
        return _pick([r for r,f in BY_MF[(model,framing)] if f.startswith(s)])
    if intent_text is not None:                      # audit 3 saw the FULL response
        t=norm(shown)
        c=[(r,fr) for r,fr,f in BY_IN.get(intent_text,[]) if f==t]
        rids={r for r,_ in c}; frs={fr for _,fr in c}
        st="unique" if len(rids)==1 else ("unmatched" if not rids else f"ambiguous_{len(rids)}")
        return (list(rids)[0] if len(rids)==1 else None), st, (FR[list(frs)[0]] if len(frs)==1 else None)
    raise ValueError

## 4 · Prompts, generations, grader labels

`grader_200_original.csv` and `grader_budget_curve.csv` are tagged with
`prompt_variant`, because they are **not** the same grader invocation:

| Variant | Response truncation | Trailing cue | Batched | Default label |
|---|---|---|---|---|
| `v1_1200char_label_cue` | 1,200 chars | `Label:` | no | HEDGE |
| `v2_untruncated_no_cue` | none | none | yes | REFUSAL |

Likewise `refused_original_scorer` (unrecovered keyword list) and
`refused_reimpl_scorer` (the list in `08_budget_curve.py`) are separate columns. They
disagree on about 1% of the original corpus and shift the refusal gap by 0.018.

In [5]:
# ---------------------------------------------------------------- corpus files
gen.assign(framing=gen.framing.map(FR)).rename(
    columns={"reply":"response","refused":"refused_original_scorer"})[
    ["response_id","model","pair_id","category","framing","response","refused_original_scorer"]
    ].to_csv(os.path.join(OUT,"generations","generations_200.csv"),index=False)

pairs.rename(columns={"eval_framing":"prompt_benchmark","deploy_framing":"prompt_user"}
    ).to_csv(os.path.join(OUT,"prompts","register_pairs_300.csv"),index=False)

for src,name,extra in [("regenerated/generations_1024_FULL.csv","generations_1024.csv",
                        ["n_new_tokens","finish_reason"]),
                       ("regenerated/generations_200_rerun.csv","generations_200_rerun.csv",
                        ["n_new_tokens","finish_reason"])]:
    d=pd.read_csv(os.path.join(DATA,src))
    d["response_id"]=[rid(m,pi,f) for m,pi,f in zip(d.model,d.pair_id,d.framing)]
    d["framing"]=d.framing.map(FR)
    d.rename(columns={"reply":"response"})[["response_id","model","pair_id","category","framing",
        "response"]+extra].to_csv(os.path.join(OUT,"generations",name),index=False)

judge["response_id"]=[rid(m,pi,f) for m,pi,f in zip(judge.model,judge.pair_id,judge.framing)]
gl=dict(zip(judge.response_id,judge.grade))
judge[["response_id","grade"]].rename(columns={"grade":"grader_label"}).assign(
    run="original_200",grader="Qwen2.5-7B-Instruct",prompt_variant="v1_1200char_label_cue"
    ).to_csv(os.path.join(OUT,"grader","grader_200_original.csv"),index=False)

bud=[]
for b in [200,384,512,768,1024]:
    d=pd.read_csv(p("graded",f"judge_B{b}.csv"))
    d["response_id"]=[rid(m,pi,f) for m,pi,f in zip(d.model,d.pair_id,d.framing)]
    bud.append(d.assign(budget=b,prompt_variant="v2_untruncated_no_cue")[
        ["response_id","budget","refused","grade","prompt_variant"]].rename(
        columns={"refused":"refused_reimpl_scorer","grade":"grader_label"}))
pd.concat(bud).to_csv(os.path.join(OUT,"grader","grader_budget_curve.csv"),index=False)

guard["response_id"]=[rid(m,pi,f) for m,pi,f in zip(guard.model,guard.pair_id,guard.framing)]
guard[["response_id","guard_label"]].rename(columns={"guard_label":"llamaguard_label"}
    ).to_csv(os.path.join(OUT,"grader","llamaguard_200.csv"),index=False)

## 5 · Audit 1 — one annotator, 500-character window, hedge only

Only 110 of 200 items resolve to a unique response, because audit 1 records the model
and framing but not which of the 300 intents it came from, and short canned replies
repeat across intents. Framing and grader label are known for all 200 regardless, which
is everything the audit-1 estimator needs.

In [6]:
# ---------------------------------------------------------------- audit 1
a1=pd.read_csv(p("audits","audit1_hedge_200.csv")); a1.columns=[c.strip() for c in a1.columns]
a1["audit_item_id"]=[f"A1-{i:04d}" for i in range(1,len(a1)+1)]
r=[resolve(t,model=m,framing=f) for t,m,f in zip(a1.reply,a1.model,a1.framing)]
a1["response_id"]=[x[0] for x in r]; a1["link_status"]=[x[1] for x in r]
A1=pd.DataFrame(dict(audit_item_id=a1.audit_item_id,response_id=a1.response_id,link_status=a1.link_status,
    framing=a1.framing.map(FR),grader_label=a1.auto_grade.map(lab),annotator="A1-1",
    label=a1.human_grade.map(lab),uncertain=pd.NA,rationale=a1.human_reason,
    display_window_chars=500,label_scheme="COMPLY/HEDGE/REFUSE"))
A1.to_csv(os.path.join(OUT,"audits","audit1_labels.csv"),index=False)

## 6 · Audit 2 — two annotators, 700-character window, stratified

The two files are asserted row-aligned first. That assertion is load-bearing: the
reviewer hypothesis that the κ = 0.09 result was a join bug rests on it, and it holds —
the files carry identical `item_id`, request, framing and response columns and differ
only in `label`.

Output is long format, one row per annotator per item, so that adding a fourth
annotator later requires no schema change.

In [7]:
# ---------------------------------------------------------------- audit 2
aA=pd.read_csv(p("audits","audit2_annotatorA.csv")); aB=pd.read_csv(p("audits","audit2_annotatorB.csv"))
for d in (aA,aB): d.columns=[c.strip() for c in d.columns]
assert (aA.item_id.values==aB.item_id.values).all(), "audit-2 files are not row-aligned"
aA["audit_item_id"]=[f"A2-{i:04d}" for i in range(1,len(aA)+1)]
aA["pair_id"]=[PROMPT2PAIR.get((norm(q),f)) for q,f in zip(aA.request,aA.framing)]
def _cands_pf(shown,pair_id,framing):
    s0=norm(shown).rstrip(); s0=s0[:max(len(s0)-3,1)]
    return [r for r,f in BY_PF[(pair_id,framing)] if f.startswith(s0)]
r=[resolve(t,pair_id=pi,framing=f) for t,pi,f in zip(aA.response,aA.pair_id,aA.framing)]
aA["response_id"]=[x[0] for x in r]; aA["link_status"]=[x[1] for x in r]
BY_F=collections.defaultdict(list)
for t in gen.itertuples(): BY_F[t.framing].append((t.response_id,t.nr))
def _grade_any(shown,pair_id,framing):
    g=_grade_set(_cands_pf(shown,pair_id,framing))
    if g: return g
    s0=norm(shown)[:300]                      # fallback: framing + 300-char prefix
    return _grade_set([r for r,f in BY_F[framing] if f.startswith(s0)])
aA["grader_label"]=[_grade_any(t,pi,f) for t,pi,f in zip(aA.response,aA.pair_id,aA.framing)]
A2=pd.concat([pd.DataFrame(dict(audit_item_id=aA.audit_item_id,response_id=aA.response_id,
    link_status=aA.link_status,framing=aA.framing.map(FR),
    grader_label=aA.grader_label,annotator=ann,label=src.map(lab),uncertain=pd.NA,
    rationale=pd.NA,display_window_chars=700,label_scheme=sc))
    for ann,src,sc in [("A2-A",aA.label,"COMPLY/HEDGE/REFUSE"),("A2-B",aB.label,"Comply/Hedge/Refusal")]],
    ignore_index=True)
A2.to_csv(os.path.join(OUT,"audits","audit2_labels.csv"),index=False)

## 7 · Audit 3 — three annotators, full response, framing-blinded

Audit 3 saw complete responses, so matching is by equality on normalised text within
the intent, not by prefix. Framing resolves for 216 of 239; the remaining 23 are the
identical canned refusal emitted under both framings. All 23 are grader-`REFUSAL` and
human-`REFUSAL`, so they affect cell counts and nothing else — but they must not be
silently dropped, hence the explicit `link_status`.

Note the shape of this audit: 419 labels over 239 items means **90 items triple-labelled
and 149 singly labelled**. Any agreement statistic computed here describes the 90.

In [8]:
# ---------------------------------------------------------------- audit 3
sh=pd.concat([pd.read_csv(p("audits",f)) for f in
              ["sheet_A1.csv","sheet_A2_labeled.csv","sheet_A3_labeled.csv"]],ignore_index=True)
sh.columns=[c.strip() for c in sh.columns]
u=sh.drop_duplicates("item_id").reset_index(drop=True)
u["audit_item_id"]=[f"A3-{i:04d}" for i in range(1,len(u)+1)]
r=[resolve(t,intent_text=q) for t,q in zip(u.model_response,u.request_description)]
u["response_id"]=[x[0] for x in r]; u["link_status"]=[x[1] for x in r]; u["framing"]=[x[2] for x in r]
# grader label resolves even when the specific response does not (all candidates agree)
def grader_for(row):
    if row.response_id in gl: return gl[row.response_id]
    c={gl[rr] for rr,fr,f in BY_IN.get(row.request_description,[]) if f==norm(row.model_response)}
    return list(c)[0] if len(c)==1 else None
u["grader_label"]=[grader_for(row) for row in u.itertuples()]
M={c:dict(zip(u.item_id,u[c])) for c in ["audit_item_id","response_id","link_status","framing","grader_label"]}
A3=pd.DataFrame(dict(audit_item_id=sh.item_id.map(M["audit_item_id"]),
    response_id=sh.item_id.map(M["response_id"]),link_status=sh.item_id.map(M["link_status"]),
    framing=sh.item_id.map(M["framing"]),grader_label=sh.item_id.map(M["grader_label"]),
    annotator="A3-"+sh.annotator.str[-1],label=sh.human_label.map(lab),uncertain=sh.uncertain,
    rationale=sh.notes,display_window_chars=pd.NA,label_scheme="HARMFUL/HEDGE/REFUSAL"))
A3.to_csv(os.path.join(OUT,"audits","audit3_labels.csv"),index=False)

## 8 · Index, integrity checks, manifest

The round-trip test worth running after this: rebuild the paper's headline numbers from
the *released* files rather than the archive. Doing so gives 0.2633, 0.0333, −0.037,
+0.272 and +0.035, against the paper's 0.263, 0.033, −0.036, +0.274, 0.035. The audit-2
values differ in the third decimal because linkage recovers grader labels for a
slightly different subset than the original script did; state that tolerance in the
release README rather than tuning it away.

In [9]:
# ---------------------------------------------------------------- index + checks
IDX=pd.concat([d.drop_duplicates("audit_item_id").assign(audit=t)[
    ["audit","audit_item_id","response_id","link_status","framing","grader_label"]]
    for t,d in [("audit1",A1),("audit2",A2),("audit3",A3)]],ignore_index=True)
IDX.to_csv(os.path.join(OUT,"audits","audited_item_index.csv"),index=False)

print("LINKAGE")
print(IDX.assign(ok=IDX.link_status.eq("unique")).groupby("audit").agg(
    items=("audit_item_id","size"),linked=("ok","sum"),
    framing_known=("framing",lambda s:s.notna().sum()),
    grader_known=("grader_label",lambda s:s.notna().sum())).to_string())
print("\nINTEGRITY")
print("  response_id unique                :",gen.response_id.is_unique,len(gen))
print("  grader rows subset of corpus      :",set(judge.response_id)<=set(gen.response_id))
print("  audit ID namespaces disjoint      :",
      not (set(A1.audit_item_id)|set(A2.audit_item_id))&set(A3.audit_item_id))
print("  audit-3 items                     :",A3.audit_item_id.nunique(),
      "| labels",len(A3),"| triple-labelled",
      int((A3.groupby('audit_item_id').size()==3).sum()))
json.dump(dict(n_pairs=len(pairs),n_responses=len(gen),models=sorted(SLUG.values()),
  framings=list(FR.values()),budgets=[200,384,512,768,1024],
  audits={"audit1":int(A1.audit_item_id.nunique()),"audit2":int(A2.audit_item_id.nunique()),
          "audit3":int(A3.audit_item_id.nunique())},
  label_sets={"audit1":1,"audit2":2,"audit3":3},
  display_window_chars={"audit1":500,"audit2":700,"audit3":None},
  scorer_variants=["original_unrecovered","reimplementation_08_budget_curve"],
  grader_prompt_variants=["v1_1200char_label_cue","v2_untruncated_no_cue"]),
  open(os.path.join(OUT,"meta","manifest.json"),"w"),indent=2)
print("\nwrote",OUT)

LINKAGE
        items  linked  framing_known  grader_known
audit                                             
audit1    200     110            200           200
audit2    442     415            442           442
audit3    239     127            216           239

INTEGRITY
  response_id unique                : True 3600
  grader rows subset of corpus      : True
  audit ID namespaces disjoint      : True
  audit-3 items                     : 239 | labels 419 | triple-labelled 90

wrote ./register-safety-gap-v2
